# Workitem Signing with APS Automation SDK

This example shows the complete signing setup for Design Automation public activities.

Python-first walkthrough for:
1. generating RSA key files,
2. uploading public key to `forgeapps/me`,
3. signing an activity id,
4. using signature in signed workitem execution.

When your automation flow uses signed/public activity execution, this setup ensures APS can verify that the workitem request was signed by your private key.

## 1) Setup

This cell loads credentials from `.env` (or environment variables).

Expected variables:
- `CLIENT_ID`
- `CLIENT_SECRET`

If either value is missing, the notebook fails early so you do not continue with partial setup.

In [ ]:
import json
import os

from dotenv import load_dotenv
from aps_automation_sdk import (
    generate_key_file,
    export_public_key,
    sign_activity,
    get_token,
    upload_public_key,
)

load_dotenv()
CLIENT_ID = os.getenv("CLIENT_ID")
CLIENT_SECRET = os.getenv("CLIENT_SECRET")
KEY_DIR = "keys"
PRIVATE_KEY_PATH = f"{KEY_DIR}/mykey.json"
PUBLIC_KEY_PATH = f"{KEY_DIR}/mypublickey.json"
os.makedirs(KEY_DIR, exist_ok=True)

if not CLIENT_ID or not CLIENT_SECRET:
    raise ValueError("Missing CLIENT_ID or CLIENT_SECRET in environment/.env")

## 2) Generate private key and export public key

This creates two files (with explicit paths you can customize):
- `PRIVATE_KEY_PATH` (default `keys/mykey.json`): private key material (keep secret, never commit)
- `PUBLIC_KEY_PATH` (default `keys/mypublickey.json`): public key that APS stores for signature verification

Use different keys per environment (dev/staging/prod) for safer rotation and isolation.

In [ ]:
saved_private = generate_key_file(PRIVATE_KEY_PATH)
export_public_key(PRIVATE_KEY_PATH, PUBLIC_KEY_PATH)
print(f"Generated private key: {saved_private}")
print(f"Generated public key: {os.path.abspath(PUBLIC_KEY_PATH)}")

## 3) Upload public key to forgeapps/me (US-East)

Here we read the public key JSON, request a 2-legged token with SDK `get_token`, and upload the public key to APS.

The response usually includes profile info such as nickname and currently configured public key.

Note (from Autodesk PATCH `forgeapps/me` docs): this call updates your forge app profile. If you upload a different public key later, it replaces the previous key (key rotation). After rotation, sign new workitems with the matching new private key.

Reference: https://aps.autodesk.com/en/docs/design-automation/v3/reference/http/forgeapps-me-PATCH/

In [ ]:
with open(PUBLIC_KEY_PATH, "r", encoding="utf-8") as f:
    public_key = json.load(f)

token = get_token(CLIENT_ID, CLIENT_SECRET)
profile = upload_public_key(token=token, public_key=public_key)

print(json.dumps(profile, indent=2))

## 4) Sign activity id

Sign the exact activity id you will execute, for example:
- `nickname.ActivityName+alias`

The signature string returned here is what you pass into signed workitem execution.

In [ ]:
activity_id = "yourNickname.BuildStructureActivity+prod"
signature = sign_activity(PRIVATE_KEY_PATH, activity_id)
print(signature)

## 5) Use signature in signed workitem execution

Use the returned signature with your existing `WorkItemAcc.run_public_activity(...)` flow.

This keeps your existing ACC orchestration unchanged; only the activity signature input is added.

In [ ]:
# Example usage (assuming you already built workitem_acc and token_3lo):
# workitem_id = workitem_acc.run_public_activity(
#     token3lo=token_3lo,
#     activity_signature=signature,
# )
# print(workitem_id)

## Optional CLI parity

The same steps can be done from CLI if you prefer shell workflows.
CLI reads `CLIENT_ID` and `CLIENT_SECRET` from env/.env and gets token automatically.

```bash
aps-automation signing generate --keyfile keys/mykey.json
aps-automation signing export --keyfile keys/mykey.json --pubkeyfile keys/mypublickey.json
aps-automation public-key upload --pubkeyfile keys/mypublickey.json
aps-automation signing sign --keyfile keys/mykey.json --activity-id "yourNickname.BuildStructureActivity+prod"
```